# Módulo 01 · Aula 2: Embeddings

**Duração:** 45 min

## Objetivos de aprendizagem

Ao final desta aula você será capaz de:

- explicar o que é um embedding e por que textos com significado parecido viram vetores próximos;
- usar `embed_query` (pergunta) e `embed_documents` (trechos) e saber por que a distinção existe;
- calcular a **similaridade de cosseno** à mão, com numpy, e interpretar o resultado;
- estimar custo e memória dos embeddings e reaproveitar vetores já calculados;
- usar o parâmetro `dimensions` (`EMBEDDINGS_DIMENSIONS`) e aplicar a regra de ouro: **um índice só funciona com o
  modelo de embeddings que o criou**;
- usar `get_embeddings()` e `describe_embeddings()` de [src/models.py](../src/models.py).

## Pré-requisitos

- Aula 1.1 (limites do LLM, RAG de brinquedo e a falha da busca por palavras).
- Deployment de embeddings no Azure OpenAI (`AZURE_OPENAI_EMBEDDINGS`, padrão `text-embedding-3-large`).

## O que vamos construir

Na aula anterior, a busca por palavras não percebeu que "me devolvem o dinheiro" e "reembolso" falam da mesma coisa.
Agora vamos transformar frases em **vetores** e medir a distância entre elas:

```mermaid
flowchart LR
    T1["'Quero devolver o fone'"] --> E[modelo de<br/>embeddings]
    T2["'Como peço o reembolso?'"] --> E
    E --> V1["[0.012, -0.034, ...]<br/>3072 números"]
    E --> V2["[0.009, -0.028, ...]<br/>3072 números"]
    V1 --> C{similaridade<br/>de cosseno}
    V2 --> C
    C --> S([próximos ou distantes?])
```

> 💰 **Economia:** calculamos os vetores **uma vez** e reutilizamos as variáveis; toda a parte de numpy roda sem
> chamadas à API.

In [1]:
# Os notebooks rodam a partir da raiz do projeto: é lá que ficam o .env, a pasta data/ e o pacote src/.
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Chat e embeddings vêm do Azure OpenAI configurado no .env (veja src/config.py).
from src.config import settings

## 1. O que é um embedding

Um **modelo de embeddings** recebe um texto e devolve uma lista de números de tamanho fixo: um **vetor**. Ele não gera
texto; só **representa** o texto, de modo que textos com significado parecido fiquem com vetores próximos. No projeto,
o cliente vem de `get_embeddings()` (seção 7), assim como o chat vem de `get_llm()`.

In [2]:
import numpy as np

from src.models import get_embeddings

embeddings = get_embeddings()

query = "Em quantos dias posso trocar um produto?"
query_vector = embeddings.embed_query(query)

print("Tipo:", type(query_vector).__name__, "de", type(query_vector[0]).__name__)
print("Dimensão:", len(query_vector))
print("Primeiros 5 valores:", [round(x, 4) for x in query_vector[:5]])
print("Norma (comprimento) do vetor:", round(float(np.linalg.norm(query_vector)), 4))

2026-09-27 22:33:43 | INFO     | src.models | Inicializando embeddings | provedor=azure | deployment=text-embedding-3-large


Tipo: list de float
Dimensão: 3072
Primeiros 5 valores: [-0.0509, -0.0051, -0.0134, 0.0241, -0.0245]
Norma (comprimento) do vetor: 0.9999


**Observe:**

- **3072** números: a dimensão nativa do `text-embedding-3-large`. Uma palavra ou uma página viram um vetor desse
  mesmo tamanho.
- Nenhuma posição isolada tem significado legível: o sentido está na **direção** do vetor inteiro.
- A norma é 1, a menos de arredondamentos: os vetores da OpenAI já vêm **normalizados**, o que simplifica a conta da
  seção 3.

## 2. `embed_query` x `embed_documents`

| Método | Recebe | Devolve | Usado para |
|---|---|---|---|
| `embed_query(text)` | um texto | um vetor | a **pergunta**, na hora da consulta |
| `embed_documents(texts)` | uma lista de textos | uma lista de vetores (mesma ordem) | os **trechos**, na ingestão |

`embed_documents` envia vários textos por requisição, bem mais rápido que `embed_query` em um laço. Vamos calcular, de
uma vez, os vetores de oito frases de clientes. A frase `exchange_days` é a mesma que enviamos com `embed_query`: os
dois vetores serão iguais?

In [3]:
import time

PHRASES = {
    "return": "Quero devolver o fone que comprei.",
    "refund": "Como peço o reembolso do meu pedido?",
    "exchange_days": "Em quantos dias posso trocar um produto?",
    "exchange_deadline": "Qual o prazo para troca de mercadoria?",
    "free_shipping": "A partir de qual valor o frete é grátis?",
    "delivery_cost": "Quanto custa a entrega para o meu CEP?",
    "battery": "Quanto tempo dura a bateria do Fone Nova Buds?",
    "cake": "Receita de bolo de cenoura com cobertura de chocolate.",
}

start = time.perf_counter()
phrase_vectors = embeddings.embed_documents(list(PHRASES.values()))  # UMA chamada para as oito frases
elapsed = time.perf_counter() - start

vectors = dict(zip(PHRASES, phrase_vectors))  # guardamos para reutilizar no resto da aula
print(f"{len(phrase_vectors)} vetores de {len(phrase_vectors[0])} dimensões em {elapsed:.2f} s")

same_text_similarity = float(np.dot(query_vector, vectors["exchange_days"]))
print(f"embed_query x embed_documents, mesmo texto: {same_text_similarity:.3f}")

8 vetores de 3072 dimensões em 0.51 s
embed_query x embed_documents, mesmo texto: 1.000


**Observe:** praticamente 1: nos modelos da OpenAI, os dois métodos fazem a mesma operação. Existem dois porque
**outros modelos tratam pergunta e documento de forma diferente** (prefixos como `"query: "` e `"passage: "`, ou um
parâmetro de tipo de entrada). Usar o método certo deixa o código correto para qualquer modelo; na prática, o *vector
store* da próxima aula faz isso por você.

## 3. Similaridade de cosseno, à mão

A medida mais usada com embeddings de texto é o cosseno do ângulo entre os dois vetores:

$$\text{cos}(a, b) = \frac{a \cdot b}{\lVert a \rVert \, \lVert b \rVert}$$

Perto de 1, mesma direção (significados parecidos); perto de 0, assuntos sem relação; negativo, direções opostas (raro
com texto). Com vetores de norma 1, o cosseno é só o **produto escalar**, então a matriz de similaridade entre as oito
frases é uma multiplicação de matrizes, sem nenhuma chamada à API.

In [4]:
import pandas as pd


def cosine_similarity(a, b) -> float:
    """Cosseno do ângulo entre dois vetores: produto escalar dividido pelo produto das normas."""
    a, b = np.asarray(a), np.asarray(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


print("return x refund, pela fórmula completa:", round(cosine_similarity(vectors["return"], vectors["refund"]), 3))

matrix = np.array(phrase_vectors)
similarity = pd.DataFrame(matrix @ matrix.T, index=list(PHRASES), columns=list(PHRASES))  # vetores de norma 1
print(similarity.round(2).to_string())

return x refund, pela fórmula completa: 0.506
                   return  refund  exchange_days  exchange_deadline  free_shipping  delivery_cost  battery  cake
return               1.00    0.51           0.50               0.44           0.25           0.26     0.39  0.22
refund               0.51    1.00           0.46               0.45           0.37           0.43     0.22  0.22
exchange_days        0.50    0.46           1.00               0.85           0.34           0.31     0.35  0.21
exchange_deadline    0.44    0.45           0.85               1.00           0.37           0.32     0.32  0.20
free_shipping        0.25    0.37           0.34               0.37           1.00           0.56     0.23  0.21
delivery_cost        0.26    0.43           0.31               0.32           0.56           1.00     0.22  0.25
battery              0.39    0.22           0.35               0.32           0.23           0.22     1.00  0.15
cake                 0.22    0.22           0.21  

**Observe:**

- O par mais próximo (0,85) é `exchange_days` x `exchange_deadline`: "em quantos dias posso trocar" e "prazo para
  troca" dizem a mesma coisa com palavras diferentes.
- Depois vêm frete grátis x custo da entrega (0,56) e devolver x reembolso (0,51). "Quero **devolver** o fone que
  comprei" e "Como peço o **reembolso** do meu pedido?" não têm nenhuma palavra relevante em comum: a busca por palavras
  da aula anterior daria zero.
- A receita de bolo fica longe de tudo (0,15 a 0,25): é o "controle" do experimento.

> ⚠️ **Armadilha comum:** ler o valor como porcentagem ("0,6 é 60% parecido"). A escala depende do modelo: no
> `text-embedding-3-large`, até frases sem relação ficam acima de zero. O que importa é a **ordem**.

### 3.1 Uma busca semântica de brinquedo

A pergunta que a busca por palavras errou na aula anterior vira um vetor, e ordenamos as frases pela similaridade. É o
que um índice vetorial faz, com milhares de trechos.

In [5]:
customer_question = "Comprei e me arrependi. Vocês me devolvem o dinheiro?"
customer_vector = embeddings.embed_query(customer_question)

ranking = sorted(
    ((cosine_similarity(customer_vector, vector), name) for name, vector in vectors.items()),
    reverse=True,
)
print(customer_question)
for score, name in ranking:
    print(f"  {score:.3f}  {PHRASES[name]}")

Comprei e me arrependi. Vocês me devolvem o dinheiro?
  0.611  Quero devolver o fone que comprei.
  0.567  Como peço o reembolso do meu pedido?
  0.457  Em quantos dias posso trocar um produto?
  0.425  Qual o prazo para troca de mercadoria?
  0.340  Quanto custa a entrega para o meu CEP?
  0.300  A partir de qual valor o frete é grátis?
  0.244  Quanto tempo dura a bateria do Fone Nova Buds?
  0.233  Receita de bolo de cenoura com cobertura de chocolate.


**Observe:** sem nenhuma palavra em comum com "reembolso", as duas frases mais próximas são devolver o fone e pedir o
reembolso; as de troca vêm logo depois (assunto vizinho), e bateria e bolo ficam no fim.

## 4. Custo e desempenho

### 4.1 Tokens e memória

Embeddings são cobrados por **token de entrada**. O `text-embedding-3-large` usa o tokenizador `cl100k_base`, que roda
localmente com o `tiktoken`. Cada número do vetor ocupa 4 bytes (`float32`).

In [6]:
from pathlib import Path

import tiktoken

encoding = tiktoken.encoding_for_model("text-embedding-3-large")
print("Tokenizador:", encoding.name)

phrase_tokens = {name: len(encoding.encode(text)) for name, text in PHRASES.items()}
print("Tokens por frase:", phrase_tokens, "| total:", sum(phrase_tokens.values()))

policies_dir = Path(settings.documents_dir) / "policies"
policy_tokens = sum(len(encoding.encode(p.read_text(encoding="utf-8"))) for p in policies_dir.glob("*.md"))
print(f"As 8 políticas inteiras: {policy_tokens} tokens")
print(f"Uma chamada com 8 frases levou {elapsed:.2f} s (medido na seção 2)")

bytes_per_vector = 3072 * 4  # float32
print(f"\nMemória: um vetor = {bytes_per_vector / 1024:.0f} KB | "
      f"índice do projeto (321 trechos) = {321 * bytes_per_vector / 1024**2:.1f} MB | "
      f"um milhão de trechos = {1_000_000 * bytes_per_vector / 1024**3:.1f} GB")

Tokenizador: cl100k_base
Tokens por frase: {'return': 11, 'refund': 12, 'exchange_days': 11, 'exchange_deadline': 12, 'free_shipping': 11, 'delivery_cost': 12, 'battery': 15, 'cake': 16} | total: 100
As 8 políticas inteiras: 10592 tokens
Uma chamada com 8 frases levou 0.51 s (medido na seção 2)

Memória: um vetor = 12 KB | índice do projeto (321 trechos) = 3.8 MB | um milhão de trechos = 11.4 GB


**Observe:** uma frase de cliente tem cerca de 10 tokens; as oito políticas, uns 10 mil. Indexar a base custa algumas
dezenas de milhares de tokens, **uma vez**; cada pergunta custa um embedding de ~10 tokens. No custo de um RAG, o peso
está no LLM e no tamanho do contexto.

| Operação | Quando acontece | Custo típico |
|---|---|---|
| embeddings dos trechos | na ingestão (e a cada documento novo ou alterado) | dezenas de milhares de tokens |
| embedding da pergunta | a cada pergunta | ~10 a 30 tokens |
| chamada ao LLM com os trechos | a cada pergunta | milhares de tokens de entrada |

### 4.2 Calcule uma vez, reutilize sempre

O mesmo texto gera (praticamente) o mesmo vetor: recalcular é pagar de novo. Em um sistema maior, um **cache** manda
para a API só os textos novos.

In [7]:
embedding_cache: dict[str, list[float]] = {text: vectors[name] for name, text in PHRASES.items()}
api_calls = 0


def embed_with_cache(texts: list[str]) -> list[list[float]]:
    """Calcula embeddings só dos textos que ainda não estão no cache."""
    global api_calls
    missing = [t for t in texts if t not in embedding_cache]
    if missing:
        api_calls += 1
        embedding_cache.update(zip(missing, embeddings.embed_documents(missing)))
    return [embedding_cache[t] for t in texts]


batch = [PHRASES["return"], PHRASES["refund"], PHRASES["battery"], "O Pix tem desconto?"]
embed_with_cache(batch)
print(f"1a rodada: {len(batch)} textos pedidos, {api_calls} chamada à API (só o texto novo foi enviado)")
embed_with_cache(batch)
print(f"2a rodada: mesmos textos, chamadas acumuladas = {api_calls} (tudo veio do cache)")

1a rodada: 4 textos pedidos, 1 chamada à API (só o texto novo foi enviado)
2a rodada: mesmos textos, chamadas acumuladas = 1 (tudo veio do cache)


> 💡 **Dica:** no projeto, o próprio **índice** é o cache: os trechos viram vetores uma vez e ficam em
> `data/faiss_index`. Na aula 1.5, a indexação incremental calcula embeddings só dos trechos novos, pelos ids estáveis.

## 5. O parâmetro `dimensions`: tamanho x qualidade

Os modelos `text-embedding-3` foram treinados de forma que as **primeiras** posições concentrem a maior parte da
informação (*Matryoshka*). Por isso a API aceita `dimensions`, que devolve um vetor mais curto; no projeto ele vem de
`EMBEDDINGS_DIMENSIONS` (vazio = 3072). Pedir `dimensions=256` equivale a **cortar** o vetor nas 256 primeiras posições
e **normalizar** de novo. Vamos simular com os vetores que já temos:

In [8]:
def truncate(vector, size: int) -> np.ndarray:
    """Corta o vetor nas `size` primeiras posições e normaliza (norma 1) de novo."""
    short = np.asarray(vector[:size])
    return short / np.linalg.norm(short)


def nearest_neighbors(size: int) -> dict[str, str]:
    """Para cada frase, a frase mais próxima usando vetores com `size` dimensões."""
    short = np.array([truncate(v, size) for v in phrase_vectors])
    sims = short @ short.T
    np.fill_diagonal(sims, -1)  # ignora a própria frase
    names = list(PHRASES)
    return {names[i]: names[int(sims[i].argmax())] for i in range(len(names))}


full = nearest_neighbors(3072)
table = pd.DataFrame({size: nearest_neighbors(size) for size in [3072, 1024, 256, 64, 16]})
print("Vizinho mais próximo de cada frase, por número de dimensões:")
print(table.to_string())

Vizinho mais próximo de cada frase, por número de dimensões:
                                3072               1024               256                64                 16  
return                        refund      exchange_days      exchange_days            battery  exchange_deadline
refund                        return             return             return      delivery_cost      free_shipping
exchange_days      exchange_deadline  exchange_deadline  exchange_deadline  exchange_deadline  exchange_deadline
exchange_deadline      exchange_days      exchange_days      exchange_days      exchange_days      exchange_days
free_shipping          delivery_cost      delivery_cost      delivery_cost      delivery_cost      delivery_cost
delivery_cost          free_shipping      free_shipping      free_shipping      free_shipping      free_shipping
battery                       return             return             return             return             return
cake                   delivery_cos

**Observe:**

- Os pares com vizinho claro (`exchange_days` x `exchange_deadline`, `free_shipping` x `delivery_cost`) resistem até
  16 dimensões.
- O caso apertado muda primeiro: `return` estava quase empatado entre `refund` (0,51) e `exchange_days` (0,50), e com
  1024 dimensões o vizinho já troca. Com 64 e 16 aparecem vizinhos sem sentido (`return` ao lado de `battery`).

Para confirmar que cortar e normalizar é o mesmo que pedir menos dimensões, fazemos **uma** chamada com
`dimensions=256`, com um cliente criado à mão só para esta demonstração (no projeto, o valor vem do `.env`).

In [9]:
from langchain_openai import AzureOpenAIEmbeddings

small_embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=settings.azure_openai_endpoint,
    api_key=settings.azure_openai_key,
    azure_deployment=settings.azure_openai_embeddings,
    api_version=settings.azure_openai_api_version,
    dimensions=256,
)
small_vector = small_embeddings.embed_query(query)  # a mesma pergunta da seção 1

print("Dimensão pedida à API:", len(small_vector))
print("Cosseno entre o vetor de 256 da API e o de 3072 cortado:",
      round(cosine_similarity(small_vector, truncate(query_vector, 256)), 4))

Dimensão pedida à API: 256
Cosseno entre o vetor de 256 da API e o de 3072 cortado: 1.0


| `dimensions` | Por vetor | 321 trechos | 1 milhão de trechos | Qualidade da busca |
|---|---|---|---|---|
| 3072 (padrão) | 12 KB | 3,8 MB | 11,4 GB | a melhor do modelo |
| 1024 | 4 KB | 1,3 MB | 3,8 GB | muito próxima |
| 256 | 1 KB | 0,3 MB | 1 GB | boa para muitos casos; perde nuances |

Para algumas centenas de trechos, 3072 dimensões não pesam nada, e o projeto usa o padrão. Reduzir faz sentido com
milhões de trechos, e a decisão deve vir de uma **avaliação** (aula 3.4).

## 6. A regra de ouro: um índice, um modelo

Um vetor só faz sentido **comparado com vetores do mesmo modelo** e da mesma dimensão: cada modelo cria o seu
próprio "sistema de coordenadas". Há dois jeitos de quebrar isso:

1. **Dimensões diferentes:** a conta nem fecha.
2. **Mesma dimensão, outro modelo:** a conta fecha, mas o resultado não significa nada, e sem erro. Para simular um
   "outro modelo" sem gastar nada, embaralhamos as posições dos vetores com uma permutação fixa.

In [10]:
try:
    cosine_similarity(query_vector, small_vector)  # 3072 x 256
except ValueError as error:
    print("1. Dimensões diferentes:", error)

rng = np.random.default_rng(seed=42)
other_model = rng.permutation(3072)  # "modelo B": mesmas 3072 dimensões, outra ordem

print("\n2. Pergunta com o modelo A, frases com o modelo B:")
mixed = sorted(
    ((cosine_similarity(customer_vector, np.asarray(vector)[other_model]), name) for name, vector in vectors.items()),
    reverse=True,
)
for score, name in mixed:
    print(f"  {score:+.3f}  {PHRASES[name]}")

1. Dimensões diferentes: shapes (3072,) and (256,) not aligned: 3072 (dim 0) != 256 (dim 0)

2. Pergunta com o modelo A, frases com o modelo B:
  -0.006  Receita de bolo de cenoura com cobertura de chocolate.
  -0.028  Como peço o reembolso do meu pedido?
  -0.028  A partir de qual valor o frete é grátis?
  -0.030  Qual o prazo para troca de mercadoria?
  -0.035  Quero devolver o fone que comprei.
  -0.040  Quanto custa a entrega para o meu CEP?
  -0.041  Quanto tempo dura a bateria do Fone Nova Buds?
  -0.042  Em quantos dias posso trocar um produto?


**Observe:** todas as similaridades ficam perto de zero e a ordem vira sorteio. Um índice criado com um modelo e
consultado com outro devolve trechos aleatórios, **sem nenhum erro**.

> ⚠️ Trocar de modelo de embeddings, de provedor ou de `EMBEDDINGS_DIMENSIONS` exige **reindexar** todos os
> documentos. Por isso o projeto grava o modelo usado ao lado do índice (`index_info.json`).

## 7. Do notebook para o projeto

### 7.1 [src/models.py](../src/models.py): `get_embeddings()` e `describe_embeddings()`

| Função | O que faz |
|---|---|
| `get_embeddings()` | cria (uma única vez, `lru_cache`) o cliente de `EMBEDDINGS_PROVIDER`: `azure` (padrão, `AzureOpenAIEmbeddings`) ou `openai` (`OpenAIEmbeddings`) |
| `describe_embeddings()` | provedor e modelo, ex.: `azure · text-embedding-3-large`; é o que fica registrado no índice |

Decisões de design:

- **Nenhum outro módulo cria um cliente de embeddings**: ingestão, busca e notebooks usam `get_embeddings()`, então
  índice e perguntas usam sempre o mesmo modelo.
- **`dimensions=settings.embeddings_dimensions`** vale para a ingestão e para a consulta ao mesmo tempo.
- **`max_retries`** vem de `settings`: falhas temporárias (429, 5xx) são repetidas automaticamente.

In [11]:
from src.models import describe_embeddings
from src.rag.vectorstore import index_info

print("Embeddings em uso:", describe_embeddings())
print("Mesma instância (cache)?", embeddings is get_embeddings())
print("Dimensões configuradas:", settings.embeddings_dimensions or "nativa (3072)")

info = index_info()  # lê data/faiss_index/index_info.json, sem carregar o índice
print("\nRegistro do índice do projeto:", info)
print("Índice compatível com o modelo atual?", info.get("embeddings") == describe_embeddings())

Embeddings em uso: azure · text-embedding-3-large
Mesma instância (cache)? True
Dimensões configuradas: nativa (3072)

Registro do índice do projeto: {'embeddings': 'azure · text-embedding-3-large', 'chunks': 321, 'dimensions': 3072, 'updated_at': '2026-09-27T20:39:15'}
Índice compatível com o modelo atual? True


**Observe:** o índice principal (`data/faiss_index`, criado pela ingestão das aulas 1.4 e 1.5) registra o modelo e a
dimensão com que foi criado, e bate com `describe_embeddings()`. Na próxima aula, `load_index()` faz essa checagem
sozinho.

## Resumo

- Um **embedding** é um vetor de tamanho fixo (3072 no `text-embedding-3-large`) que representa o significado de um
  texto. Textos parecidos viram vetores próximos, mesmo sem palavras em comum.
- `embed_query` é para a pergunta e `embed_documents` para os trechos (em lote).
- **Similaridade de cosseno** = produto escalar dividido pelas normas; com vetores normalizados, só o produto escalar.
  O que importa é a **ordem**, não o valor absoluto.
- Embeddings custam por token de entrada: indexar é caro e raro, a pergunta é barata. Calcule uma vez e reutilize.
- `dimensions` corta o vetor: menos memória, alguma perda de qualidade. Decida com avaliação.
- **Regra de ouro:** o índice só funciona com o modelo (e a dimensão) que o criou. Trocar exige reindexar.

## Próxima aula

Na aula 1.3 (`01_03_vector_search`) vamos guardar vetores em um **índice vetorial**: o `InMemoryVectorStore` do
langchain-core e depois o **FAISS**, com distância L2, `k`, filtro por metadados e índice salvo em disco.